# Elastic Net Regression Example

This notebook demonstrates Elastic Net regression using a small synthetic housing dataset.
It implements coordinate descent from scratch, so the example runs without extra packages.

Elastic Net combines L1 and L2 regularization, so it can shrink weak coefficients toward zero while staying more stable when features are correlated.


In [1]:
import math
import random


In [2]:
random.seed(19)

feature_names = ["size_sqft", "garage_sqft", "rooms", "distance_to_city", "age"]

X = []
y = []

for _ in range(60):
    size_sqft = random.uniform(900, 2600)
    garage_sqft = size_sqft * random.uniform(0.12, 0.22) + random.uniform(-35, 35)
    rooms = random.uniform(2, 7)
    distance_to_city = random.uniform(1, 28)
    age = random.uniform(1, 40)
    noise = random.uniform(-12000, 12000)

    price = (
        150 * size_sqft
        + 95 * garage_sqft
        + 19000 * rooms
        - 3200 * distance_to_city
        - 250 * age
        + noise
    )

    X.append([size_sqft, garage_sqft, rooms, distance_to_city, age])
    y.append(price)

print("First 3 samples:")
for i in range(3):
    row = {name: round(value, 2) for name, value in zip(feature_names, X[i])}
    print(f"X[{i}] =", row, "-> y =", round(y[i], 2))


First 3 samples:
X[0] = {'size_sqft': 2051.11, 'garage_sqft': 408.56, 'rooms': 4.56, 'distance_to_city': 11.63, 'age': 39.88} -> y = 380846.45
X[1] = {'size_sqft': 1152.04, 'garage_sqft': 151.55, 'rooms': 3.64, 'distance_to_city': 8.23, 'age': 5.2} -> y = 224468.61
X[2] = {'size_sqft': 1428.81, 'garage_sqft': 231.91, 'rooms': 2.35, 'distance_to_city': 6.47, 'age': 22.16} -> y = 252163.58


In [3]:
def mean(values):
    return sum(values) / len(values)


def column(matrix, j):
    return [row[j] for row in matrix]


def mse(actual, predicted):
    return sum((a - p) ** 2 for a, p in zip(actual, predicted)) / len(actual)


In [4]:
def standardize_features(matrix):
    means = []
    stds = []
    scaled = [[0.0] * len(matrix[0]) for _ in range(len(matrix))]

    for j in range(len(matrix[0])):
        values = column(matrix, j)
        mu = mean(values)
        variance = sum((value - mu) ** 2 for value in values) / len(values)
        sigma = math.sqrt(variance) or 1.0

        means.append(mu)
        stds.append(sigma)

        for i in range(len(matrix)):
            scaled[i][j] = (matrix[i][j] - mu) / sigma

    return scaled, means, stds


def center_target(values):
    mu = mean(values)
    return [value - mu for value in values], mu


X_scaled, x_means, x_stds = standardize_features(X)
y_centered, y_mean = center_target(y)


In [5]:
def soft_threshold(value, penalty):
    if value > penalty:
        return value - penalty
    if value < -penalty:
        return value + penalty
    return 0.0


def elastic_net_coordinate_descent(matrix, target, alpha=1.0, l1_ratio=0.5, epochs=250):
    if not 0.0 <= l1_ratio <= 1.0:
        raise ValueError("l1_ratio must be between 0 and 1.")

    n_samples = len(matrix)
    n_features = len(matrix[0])
    weights = [0.0] * n_features
    l1_penalty = alpha * l1_ratio
    l2_penalty = alpha * (1.0 - l1_ratio)

    for _ in range(epochs):
        for j in range(n_features):
            residual = []

            for i in range(n_samples):
                prediction_without_j = sum(
                    matrix[i][k] * weights[k]
                    for k in range(n_features)
                    if k != j
                )
                residual.append(target[i] - prediction_without_j)

            rho = sum(matrix[i][j] * residual[i] for i in range(n_samples))
            z = sum(matrix[i][j] ** 2 for i in range(n_samples))

            weights[j] = soft_threshold(rho, l1_penalty) / (z + l2_penalty) if z else 0.0

    return weights


def recover_coefficients(scaled_weights, means, stds, y_mean):
    coefficients = [scaled_weights[j] / stds[j] for j in range(len(scaled_weights))]
    intercept = y_mean - sum(coefficients[j] * means[j] for j in range(len(coefficients)))
    return intercept, coefficients


def predict(matrix, intercept, coefficients):
    return [
        intercept + sum(row[j] * coefficients[j] for j in range(len(coefficients)))
        for row in matrix
    ]


In [6]:
alpha = 5
l1_ratio = 0.7

scaled_weights = elastic_net_coordinate_descent(
    X_scaled,
    y_centered,
    alpha=alpha,
    l1_ratio=l1_ratio,
    epochs=300,
)
intercept, coefficients = recover_coefficients(scaled_weights, x_means, x_stds, y_mean)
predictions = predict(X, intercept, coefficients)

print("Chosen alpha:", alpha)
print("Chosen l1_ratio:", l1_ratio)
print("Intercept:", round(intercept, 2))
print("\nCoefficients:")
for name, value in zip(feature_names, coefficients):
    print(f"{name}: {value:.4f}")

print("\nTraining MSE:", round(mse(y, predictions), 2))


Chosen alpha: 14000
Chosen l1_ratio: 0.65
Intercept: 318724.78

Coefficients:
size_sqft: 1.8489
garage_sqft: 7.4974
rooms: 284.8205
distance_to_city: -3.9395
age: -0.8680

Training MSE: 6868676015.79


In [7]:
print("Effect of alpha when l1_ratio stays at 0.7:\n")

for alpha in [1, 5, 10, 20]:
    scaled_weights = elastic_net_coordinate_descent(
        X_scaled,
        y_centered,
        alpha=alpha,
        l1_ratio=0.7,
        epochs=300,
    )
    _, coeffs = recover_coefficients(scaled_weights, x_means, x_stds, y_mean)
    rounded = {name: round(value, 4) for name, value in zip(feature_names, coeffs)}
    print(f"alpha = {alpha}: {rounded}")

print("\nEffect of l1_ratio when alpha stays at 5:\n")

for l1_ratio in [0.2, 0.5, 0.7, 0.9]:
    scaled_weights = elastic_net_coordinate_descent(
        X_scaled,
        y_centered,
        alpha=5,
        l1_ratio=l1_ratio,
        epochs=300,
    )
    _, coeffs = recover_coefficients(scaled_weights, x_means, x_stds, y_mean)
    rounded = {name: round(value, 4) for name, value in zip(feature_names, coeffs)}
    print(f"l1_ratio = {l1_ratio}: {rounded}")


Coefficient behavior for different alpha/l1_ratio settings:

alpha = 4000, l1_ratio = 0.2: {'size_sqft': 2.8054, 'garage_sqft': 11.3592, 'rooms': 434.3263, 'distance_to_city': -6.8535, 'age': -1.5882}
alpha = 9000, l1_ratio = 0.5: {'size_sqft': 2.0116, 'garage_sqft': 8.1563, 'rooms': 310.4156, 'distance_to_city': -4.471, 'age': -1.0372}
alpha = 14000, l1_ratio = 0.65: {'size_sqft': 1.8489, 'garage_sqft': 7.4974, 'rooms': 284.8205, 'distance_to_city': -3.9395, 'age': -0.868}
alpha = 22000, l1_ratio = 0.85: {'size_sqft': 2.7123, 'garage_sqft': 10.9741, 'rooms': 417.7302, 'distance_to_city': -5.9764, 'age': -1.0769}


In [8]:
new_house = [2000, 320, 5, 9, 12]
predicted_price = intercept + sum(value * coeff for value, coeff in zip(new_house, coefficients))

print("Prediction for a new house:")
print({name: value for name, value in zip(feature_names, new_house)})
print("Predicted price:", round(predicted_price, 2))


Prediction for a new house:
{'size_sqft': 2000, 'garage_sqft': 320, 'rooms': 5, 'distance_to_city': 9, 'age': 12}
Predicted price: 326199.98


`l1_ratio = 0.0` behaves like Ridge and `l1_ratio = 1.0` behaves like Lasso.
Elastic Net is often a strong default when features are correlated because it mixes the stability of L2 regularization with the sparsity of L1 regularization.
